# NeuroScan — Model Evaluation & Comparison
Loads all four trained models, runs evaluation on the full test set (400 × 4 = 1600 samples),
and exports metrics to JSON and CSV.

**Before running:**
1. Attach the Brain Tumor MRI Dataset (masoudnickparvar/brain-tumor-mri-dataset)
2. Attach your neuroscan weights dataset (upload a dataset containing the four `.pt` files:
   `efficientnet_b3_neuroscan.pt`, `vgg16_neuroscan.pt`,
   `densenet121_neuroscan.pt`, `inception_v3_neuroscan.pt`)
   — OR run this notebook in the same session as the training notebooks so weights
   exist under `/kaggle/working/neuroscan/`.

In [ ]:
import subprocess, torch

r = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU       :', r.stdout.strip() or 'none')
print('PyTorch   :', torch.__version__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device    : {device}')

In [ ]:
import os

# ── Locate dataset ────────────────────────────────────────────────────────────
def find_dataset_root(base='/kaggle/input'):
    for root, dirs, _ in os.walk(base):
        if 'Training' in dirs and 'Testing' in dirs:
            return root
    return None

DATA_DIR = find_dataset_root()
if DATA_DIR is None:
    raise FileNotFoundError('Dataset not found — attach masoudnickparvar/brain-tumor-mri-dataset')
print(f'Dataset   : {DATA_DIR}')

# ── Locate weights ─────────────────────────────────────────────────────────────
# Search /kaggle/input/ first (attached dataset), then /kaggle/working/ (same session)
WEIGHT_FILES = {
    'efficientnet_b3': 'efficientnet_b3_neuroscan.pt',
    'vgg16':           'vgg16_neuroscan.pt',
    'densenet121':     'densenet121_neuroscan.pt',
    'inception_v3':    'inception_v3_neuroscan.pt',
}

def find_weight(filename):
    for search_root in ['/kaggle/input', '/kaggle/working']:
        for root, dirs, files in os.walk(search_root):
            if filename in files:
                return os.path.join(root, filename)
    return None

weight_paths = {}
for model_id, fname in WEIGHT_FILES.items():
    path = find_weight(fname)
    weight_paths[model_id] = path
    status = f'FOUND: {path}' if path else 'NOT FOUND'
    print(f'  {model_id:<20} {status}')

missing = [k for k, v in weight_paths.items() if v is None]
if missing:
    print(f'\nWARNING: missing weights for: {missing}')
    print('These models will be skipped in the comparison.')

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report)
from tqdm import tqdm

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = 4
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]


class BrainTumorDataset(Dataset):
    def __init__(self, root, transform):
        self.samples   = []
        self.label_map = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.transform = transform
        for cls in CLASS_NAMES:
            d = os.path.join(root, cls)
            if not os.path.isdir(d): continue
            for f in sorted(os.listdir(d)):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(d, f), self.label_map[cls]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, lbl = self.samples[i]
        return self.transform(Image.open(p).convert('RGB')), lbl


def make_loader(img_size):
    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    ds = BrainTumorDataset(os.path.join(DATA_DIR, 'Testing'), tfm)
    return DataLoader(ds, batch_size=32, shuffle=False, num_workers=2,
                      pin_memory=(device.type == 'cuda'))


loader_224 = make_loader(224)
loader_299 = make_loader(299)
print(f'Test samples: {len(loader_224.dataset)}  (expected 1600)')

In [ ]:
# ── Model builders — architectures must match training notebooks exactly ───────

def build_efficientnet_b3():
    m = models.efficientnet_b3(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_f, NUM_CLASSES))
    return m


def build_vgg16():
    m = models.vgg16(weights=None)
    m.classifier = nn.Sequential(
        *list(m.classifier.children())[:-1],
        nn.Dropout(p=0.4),
        nn.Linear(4096, NUM_CLASSES),
    )
    return m


def build_densenet121():
    m = models.densenet121(weights=None)
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(m.classifier.in_features, NUM_CLASSES),
    )
    return m


def build_inception_v3():
    m = models.inception_v3(weights=None, aux_logits=True)
    m.AuxLogits.fc = nn.Linear(m.AuxLogits.fc.in_features, NUM_CLASSES)
    m.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(m.fc.in_features, NUM_CLASSES))
    return m


MODEL_REGISTRY = {
    'efficientnet_b3': {'builder': build_efficientnet_b3, 'img_size': 224},
    'vgg16':           {'builder': build_vgg16,           'img_size': 224},
    'densenet121':     {'builder': build_densenet121,     'img_size': 224},
    'inception_v3':    {'builder': build_inception_v3,    'img_size': 299},
}

print('Model registry configured for 4 models.')

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbls in tqdm(loader, leave=False):
        out = model(imgs.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    return np.array(labels), np.array(preds)


def compute_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True,
                                   zero_division=0)
    per_class = {}
    for cls in CLASS_NAMES:
        per_class[cls] = {
            'precision': round(report[cls]['precision'], 4),
            'recall':    round(report[cls]['recall'],    4),
            'f1':        round(report[cls]['f1-score'],  4),
            'support':   int(report[cls]['support']),
        }
    return {
        'accuracy':            round(accuracy_score(y_true, y_pred), 4),
        'f1_weighted':         round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'precision_weighted':  round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'recall_weighted':     round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'per_class':           per_class,
    }

In [ ]:
# ── Run evaluation for each available model ────────────────────────────────────
all_results = {}

for model_id, cfg in MODEL_REGISTRY.items():
    wpath = weight_paths.get(model_id)
    if wpath is None:
        print(f'\n[SKIP] {model_id} — weights not found')
        continue

    print(f'\n{'='*60}')
    print(f'Evaluating: {model_id}')
    print(f'Weights   : {wpath}')

    m = cfg['builder']()
    state = torch.load(wpath, map_location=device)
    m.load_state_dict(state)
    m = m.to(device)

    loader = loader_299 if cfg['img_size'] == 299 else loader_224
    y_true, y_pred = evaluate_model(m, loader)

    metrics = compute_metrics(y_true, y_pred)
    all_results[model_id] = metrics

    print(f"Accuracy  : {metrics['accuracy']:.4f}")
    print(f"F1        : {metrics['f1_weighted']:.4f}")
    print(f"Precision : {metrics['precision_weighted']:.4f}")
    print(f"Recall    : {metrics['recall_weighted']:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    del m
    if device.type == 'cuda': torch.cuda.empty_cache()

print(f'\nEvaluation complete for {len(all_results)} model(s).')

In [ ]:
import json, csv
import pandas as pd

OUT_DIR = '/kaggle/working/neuroscan/results'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Save full metrics to JSON ─────────────────────────────────────────────────
json_path = os.path.join(OUT_DIR, 'model_comparison.json')
with open(json_path, 'w') as fh:
    json.dump(all_results, fh, indent=2)
print(f'JSON saved : {json_path}')

# ── Build flat CSV ─────────────────────────────────────────────────────────────
rows = []
for model_id, m in all_results.items():
    row = {
        'model':              model_id,
        'accuracy':           m['accuracy'],
        'f1_weighted':        m['f1_weighted'],
        'precision_weighted': m['precision_weighted'],
        'recall_weighted':    m['recall_weighted'],
    }
    for cls in CLASS_NAMES:
        row[f'{cls}_precision'] = m['per_class'][cls]['precision']
        row[f'{cls}_recall']    = m['per_class'][cls]['recall']
        row[f'{cls}_f1']        = m['per_class'][cls]['f1']
    rows.append(row)

df = pd.DataFrame(rows).set_index('model')

csv_path = os.path.join(OUT_DIR, 'model_comparison.csv')
df.to_csv(csv_path)
print(f'CSV saved  : {csv_path}')

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print('\n' + '='*72)
print('NEUROSCAN — MODEL COMPARISON SUMMARY')
print('Test set: 400 samples × 4 classes = 1600 total')
print('='*72)

summary_cols = ['accuracy', 'f1_weighted', 'recall_weighted',
                'glioma_recall', 'meningioma_recall', 'pituitary_recall', 'notumor_recall']
summary_df = df[summary_cols].sort_values('accuracy', ascending=False)
summary_df.columns = ['Accuracy', 'F1', 'Recall', 'Glioma R', 'Menin R', 'Pituit R', 'NoTumor R']
print(summary_df.to_string(float_format='{:.4f}'.format))

best = summary_df.index[0]
print(f'\n  Best overall : {best}  ({summary_df.loc[best, "Accuracy"]:.4f} accuracy)')
print(f'  Best recall  : {summary_df["Recall"].idxmax()}  ({summary_df["Recall"].max():.4f})')
print(f'  Best glioma  : {summary_df["Glioma R"].idxmax()}  ({summary_df["Glioma R"].max():.4f})')
print(f'  Best menin   : {summary_df["Menin R"].idxmax()}  ({summary_df["Menin R"].max():.4f})')

In [ ]:
# ── List all output files ──────────────────────────────────────────────────────
print('Output files:')
for root, dirs, files in os.walk('/kaggle/working/neuroscan/results'):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {p}  ({os.path.getsize(p)/1e3:.1f} KB)')